In [ ]:
%pip install numpy pandas

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
filmes = pd.read_csv('movies_metadata.csv', low_memory=False)
filmes.head(3)

In [ ]:
# importando arquivo de avaliações e avaliando as primiras linhas
avaliacoes = pd.read_csv('ratings.csv')
avaliacoes.head()

PROCESSAMENTOS

In [ ]:
#selecionando as variaveis de interesse do dataset de filmes
filmes = filmes[['id', 'original_title', 'original_language', 'vote_count']]

#renomeando as colunas
filmes.rename(columns={'id': 'id_filme', 'original_title': 'titulo', 'original_language': 'linguagem', 'vote_count': 'qt_avaliacoes'}, inplace=True)

#exibir
filmes.head()

In [ ]:
#fazendo a mesma coisa só que com o dataset de avaliações
avaliacoes = avaliacoes[['userId', 'movieId', 'rating']]

#renomeando as colunas
avaliacoes.rename(columns={'userId': 'id_usuario', 'movieId': 'id_filme', 'rating': 'avaliacao'}, inplace=True)

#exibir
avaliacoes.head()

In [ ]:
# verificar se tem valores nulos no dataset de filmes
filmes.isna().sum()

In [ ]:
#removendo os valores nulos do dataset de filmes
filmes.dropna(inplace=True)
#inplace serve para que a alteração seja feita no próprio dataset, sem precisar criar uma cópia

In [ ]:
#verificando se os valores nulos foram removidos
filmes.isna().sum()

In [ ]:
#verificando se tem valores nulos no dataset de avaliações
avaliacoes.isna().sum()
#não tem

In [ ]:
#quantidade de avaliações por usuário
avaliacoes['id_usuario'].value_counts()

In [ ]:
#pegar as avliacoes com mais peso
#considerar usuarios que avaliaram mais de 999 filmes
qt_avaliacoes = avaliacoes['id_usuario'].value_counts() > 999
y = qt_avaliacoes[qt_avaliacoes].index
y.shape

In [ ]:
# oque é o y
y
# usuarios selecionados que avaliaram mais de 999 filmes
# quantidade é (2509,) total de 2509 usuários que avaliaram mais de 999 filmes

In [ ]:
#tamanho do dataset de avaliações
avaliacoes.shape

In [ ]:
# Separando o conjunto de dados para avaliações que iremos utilizar para o sistema de recomendação 
# filtrando apenas os usuários que avaliaram mais de 999 filmes

avaliacoes = avaliacoes[avaliacoes['id_usuario'].isin(y)]

# o comando isin() é utilizado para filtrar os dados do DataFrame com base na lista de valores y.
# no caso o dataframe é o y que contém os usuários que avaliaram mais de 999 filmes
# e o comando avaliacoes['id_usuario'].isin(y) retorna uma série booleana indicando se cada valor da coluna 'id_usuario' está presente na lista y.

In [ ]:
#vizualizando o dataset de avaliações filtrado
avaliacoes.shape
# qual a diferença entre .shape e .head()?
# .shape retorna uma tupla com o número de linhas e colunas do DataFrame
# enquanto .head() retorna as primeiras n linhas do DataFrame (por padrão, 5 linhas).

In [ ]:
# vizualizando com .head()
avaliacoes.head()

In [ ]:
#dataframe filmes
filmes.head()

In [ ]:
# fazer o mesmo modelo de filtragem colaborativa com os filmes
# usaremos filmes que possuem mais de 999 avalizacoes para que o modelo seja mais assertivo
filmes = filmes[filmes['qt_avaliacoes'] > 999 ]

In [ ]:
# agrupar e vizualizar pela qt de linguagem dos filmes
filmes_linguagem = filmes['linguagem'].value_counts()
filmes_linguagem.head(20)

In [ ]:
filmes = filmes[filmes['linguagem'] == 'en']

In [ ]:
# tipos de dados filmes
filmes.info()

In [ ]:
# tipos de dados avaliacoes
avaliacoes.info()

In [ ]:
# mudar o tipo da variavel para int
filmes['id_filme'] = filmes['id_filme'].astype(int)


In [ ]:
#verificar
filmes.shape

In [ ]:
filmes.info()

In [ ]:
# concatenar dataframes
avaliacoes_e_filmes = avaliacoes.merge(filmes, on = 'id_filme')
avaliacoes_e_filmes.head()

In [ ]:
# verificando a quantidade
avaliacoes_e_filmes.shape

In [ ]:
# verificando se tem valor nulo
avaliacoes_e_filmes.isna().sum()

In [ ]:
avaliacoes_e_filmes.head(20)

In [ ]:
# descartar valores duplicados nas avaliacoes, para que n tenha problema do mesmo usuario avaliar varias vezes um filme
# funçao do pandas a ser utilizada ".drop_duplicates"
avaliacoes_e_filmes.drop_duplicates(['id_usuario','id_filme'], inplace = True)

In [ ]:
# vendo se houve alteração
avaliacoes_e_filmes.shape
#não tinha duplicidade

In [ ]:
# excluir a variavel id_filme, pois nao vamos utilizar
del avaliacoes_e_filmes['id_filme']

In [ ]:
# ver dataframe sem a aba id_filme
avaliacoes_e_filmes.head(20)

In [ ]:
# precisa fazer um PIVOT.queremos que cada id_usuario seja uma variavel com valor de nota
#para cada filme
filmes_pivot = avaliacoes_e_filmes.pivot_table(columns = 'id_usuario', index = 'titulo', values = 'avaliacao')

#avaliar arquivo atualizado para pivot
filmes_pivot.head(20)

In [ ]:
# preencher valores nulos com zero
filmes_pivot.fillna(0, inplace = True)
filmes_pivot.head()

In [ ]:
%pip install scipy

In [ ]:
# importar crs_matrix do pacote scipy.sparse
# isso possibilita criarmos uma matriz esparsa, que é uma representação eficiente de matrizes com muitos elementos nulos.
from scipy.sparse import csr_matrix

In [ ]:
# transformar o dataframe filmes_pivot em uma matriz esparsa
filmes_sparse = csr_matrix(filmes_pivot)

In [ ]:
# tipo de objeto de filmes_sparse
type(filmes_sparse)

In [ ]:
# importar o KNN do pacote NearestNeighbors do Scikit-learn
%pip install scikit-learn
from sklearn.neighbors import NearestNeighbors

In [ ]:
# criando e treinando o modelo preditivo KNN
modelo = NearestNeighbors(algorithm = 'brute')
modelo.fit(filmes_sparse)

ALGUMAS PREVISOES

In [ ]:
# 127 hours
distances, suggestions = modelo.kneighbors(filmes_pivot.filter(items = ['127 Hours'], axis=0).values.reshape(1, -1) )

for i in range(len(suggestions)):
    print(filmes_pivot.index[suggestions[i]])

In [ ]:
# Toy Story
distances, suggestions = modelo.kneighbors(filmes_pivot.filter(items = ['Toy Story'], axis=0).values.reshape(1, -1) )

for i in range(len(suggestions)):
    print(filmes_pivot.index[suggestions[i]])

In [ ]:
distances, suggestions = modelo.kneighbors(filmes_pivot.filter(items = ['10 Things I Hate About You'], axis=0).values.reshape(1, -1) )

for i in range(len(suggestions)):
    print(filmes_pivot.index[suggestions[i]])

In [ ]:
distances, suggestions = modelo.kneighbors(filmes_pivot.filter(items = ['Interstellar'], axis=0).values.reshape(1, -1) )

for i in range(len(suggestions)):
    print(filmes_pivot.index[suggestions[i]])